# 02 - Exploratory Data Analysis

Characterise the Backblaze Q1 2026 fleet before building any features. Every section here
exists to answer a specific design question in notebook 03 rather than to produce plots for
their own sake.

| Section | Question it answers |
|---|---|
| 2. Fleet overview | Is the panel a genuine daily time series? |
| 3. Manufacturer & model mix | What vendor families must the model generalise across? |
| 4. SMART availability matrix | Which attributes may we actually use, and for whom? |
| 5. Failure rates & bathtub curve | Does age belong in the feature set? |
| 6. Topology & tome geometry | What exactly is the safety invariant defending? |
| 7. Pre-failure trajectories | Which attributes carry usable early warning? |
| 8. Anomaly onset lead time | Is a 7-day horizon defensible? |
| 9. Censoring | Which drives must be excluded from negative labels? |
| 10. Class prevalence | How severe is the imbalance we must design metrics around? |

## One methodological rule for this notebook

The panel is a **failure-enriched sample**, so raw rates computed on it are biased upward.
Anything describing the *fleet* (failure rates, AFR, model mix) is therefore computed from
`drive_registry.parquet`, which covers every drive. The panel is used only for *telemetry*
analysis, where per-drive trajectories are what matter and enrichment is harmless.
Conflating the two is the easiest way to report a flattering number that means nothing.

In [ ]:
!pip install -q duckdb pyarrow polars

import os, warnings
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (11, 4.5), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/hpe_capstone" if IN_COLAB else os.path.expanduser("~/hpe_capstone")
PANEL_GLOB = f"{DRIVE_ROOT}/panel/panel_*.parquet"
FLEET_GLOB = f"{DRIVE_ROOT}/fleet_minimal/fleet_*.parquet"
REGISTRY   = f"{DRIVE_ROOT}/drive_registry.parquet"

panel    = pl.scan_parquet(PANEL_GLOB)          # lazy - nothing read yet
registry = pd.read_parquet(REGISTRY)

print(f"registry : {len(registry):,} drives")
print(f"panel cols: {len(panel.collect_schema().names())}")

## 2. Fleet overview

The first check repeats the validation from notebook 01 because it is the single assumption
everything else rests on. If observed-days-per-drive is not close to the window length, the
panel is not a time series and no rolling feature can be trusted.

In [ ]:
overview = panel.select(
    rows     = pl.len(),
    drives   = pl.col("serial_number").n_unique(),
    days     = pl.col("date").n_unique(),
    min_date = pl.col("date").min(),
    max_date = pl.col("date").max(),
    failures = pl.col("failure").sum(),
).collect().to_pandas().T.rename(columns={0: "value"})
print(overview.to_string())

days_per_drive = panel.group_by("serial_number").agg(n = pl.len()).collect()["n"]
print("\n=== observed days per drive (panel) ===")
print(days_per_drive.describe().to_pandas().to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].hist(days_per_drive.to_numpy(), bins=60, color="#1f7a6f", edgecolor="white", linewidth=0.4)
ax[0].set_title("Observed days per drive\n(a healthy panel is right-heavy, not left-heavy)")
ax[0].set_xlabel("days observed"); ax[0].set_ylabel("drives")

daily = (panel.group_by("date")
              .agg(drives = pl.len(), failures = pl.col("failure").sum())
              .sort("date").collect().to_pandas())
ax[1].plot(daily.date, daily.drives, color="#1f7a6f", lw=1.4)
ax[1].set_title("Cohort drives reporting per day")
ax[1].set_xlabel("date"); ax[1].set_ylabel("drives")
ax[1].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

print(f"median days/drive = {days_per_drive.median()}  "
      f"| single-day drives = {(days_per_drive == 1).sum():,}")

## 3. Manufacturer and model mix

Drive Stats has no manufacturer column, so it is derived from the model string. This matters
more than it looks: SMART attribute *meaning and availability* is vendor-specific, and the
cross-vendor generalisation test in Part 7 is organised along these families.

In [ ]:
def manufacturer_of(model: str) -> str:
    m = (model or "").upper()
    if m.startswith("ST"):                                    return "Seagate"
    if m.startswith(("WDC", "WUH", "WSH", "WD")):              return "WDC"
    if m.startswith(("HGST", "HUH", "HMS", "HDS")):            return "HGST"
    if m.startswith(("TOSHIBA", "MD", "MG", "MQ")):            return "Toshiba"
    if m.startswith(("SAMSUNG", "MZ")):                        return "Samsung"
    if m.startswith(("CT", "MICRON")):                         return "Micron"
    if m.startswith(("DELLBOSS", "SSDSC")):                     return "Intel/Dell"
    return "Other"

registry["manufacturer"] = registry.model.map(manufacturer_of)
registry["capacity_tb"]  = (registry.capacity_bytes / 1e12).round(1)

mfr = (registry.groupby("manufacturer")
       .agg(drives=("serial_number", "size"),
            failures=("ever_failed", "sum"),
            drive_days=("n_days", "sum"))
       .assign(afr_pct=lambda d: (d.failures / d.drive_days * 365 * 100).round(2))
       .sort_values("drives", ascending=False))
print("=== fleet by manufacturer (from registry, unbiased) ===")
print(mfr.to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(mfr.index, mfr.drives, color="#1f7a6f"); ax[0].set_title("Drives by manufacturer")
ax[0].tick_params(axis="x", rotation=20)
ax[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f"{v/1000:.0f}k"))
sub = mfr[mfr.drives > 500]
ax[1].bar(sub.index, sub.afr_pct, color="#c1553b"); ax[1].set_title("Annualised failure rate (%)")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 4. SMART availability matrix

**The most consequential cell in this notebook.** It measures, per manufacturer, the fraction
of rows where each SMART attribute is non-null.

The expected headline result: `smart_187` and `smart_188` are reported by Seagate and are
absent on Toshiba and WDC, producing roughly a two-thirds null rate fleet-wide. An earlier
processing attempt treated that 67% as missing data. It is not missing - those drives do not
have the sensor. Imputing a value there fabricates a reading, and imputing the *fleet mean*
additionally leaks manufacturer identity into a column that is supposed to be a measurement.

Notebook 03 therefore uses explicit missingness indicators and keeps manufacturer as a
first-class feature, so the model can learn "this attribute is unavailable here" rather than
being handed an invented number.

In [ ]:
smart_cols = [c for c in panel.collect_schema().names() if c.startswith("smart_")]
raw_cols   = [c for c in smart_cols if c.endswith("_raw")]

# Kept lazy on purpose: collecting every SMART column for ~8M rows would need several GB of
# RAM. Joining the manufacturer lookup and aggregating inside polars returns only the
# small result table.
mfr_lookup = pl.from_pandas(registry[["serial_number", "manufacturer"]])

matrix = (panel.select(["serial_number"] + raw_cols)
               .join(mfr_lookup.lazy(), on="serial_number", how="left")
               .group_by("manufacturer")
               .agg([(100 * pl.col(c).is_not_null().mean()).round(1).alias(c)
                     for c in raw_cols])
               .collect()
               .to_pandas()
               .set_index("manufacturer"))

matrix = matrix.loc[matrix.index.isin(mfr[mfr.drives > 500].index)]
matrix.columns = [c.replace("smart_", "").replace("_raw", "") for c in matrix.columns]

fig, ax = plt.subplots(figsize=(14, 3.2))
im = ax.imshow(matrix.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(matrix.columns))); ax.set_xticklabels(matrix.columns, fontsize=8)
ax.set_yticks(range(len(matrix.index)));   ax.set_yticklabels(matrix.index, fontsize=9)
ax.set_title("SMART raw attribute availability by manufacturer (% of rows non-null)")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix.values[i, j]
        if v < 99:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6.5)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01); ax.grid(False)
plt.tight_layout(); plt.show()

print("=== attributes that are NOT universally available ===")
print(matrix.loc[:, (matrix < 99).any()].to_string())

In [ ]:
print("=== fleet-wide null rate per attribute ===")
nulls = (panel.select([(100 * pl.col(c).is_null().mean()).round(2).alias(c)
                       for c in raw_cols])
              .collect().to_pandas().T[0].sort_values(ascending=False))
print(nulls[nulls > 0].to_string())

print("\nInterpretation guide:")
print("  ~67% null + concentrated in one vendor -> vendor-conditional sensor, use an indicator")
print("  low null rate, spread across vendors   -> genuine dropout, safe to forward-fill")

## 5. Failure rates, capacity and the bathtub curve

Computed on the registry, so these are true fleet rates. The bathtub curve tells us whether
drive age carries signal independent of SMART counters - if failure risk is U-shaped in age,
then age belongs in the feature set as more than a nuisance variable.

In [ ]:
top_models = (registry.groupby("model")
              .agg(drives=("serial_number", "size"),
                   failures=("ever_failed", "sum"),
                   drive_days=("n_days", "sum"))
              .query("drives >= 1000")
              .assign(afr_pct=lambda d: (d.failures / d.drive_days * 365 * 100).round(2))
              .sort_values("afr_pct", ascending=False))

print("=== highest-AFR models (>=1000 drives) ===")
print(top_models.head(12).to_string())
print("\n=== lowest-AFR models ===")
print(top_models.tail(8).to_string())

In [ ]:
cap = (registry[registry.capacity_tb > 0]
       .groupby("capacity_tb")
       .agg(drives=("serial_number", "size"),
            failures=("ever_failed", "sum"),
            drive_days=("n_days", "sum"))
       .query("drives >= 500")
       .assign(afr_pct=lambda d: (d.failures / d.drive_days * 365 * 100).round(2)))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(cap.index.astype(str), cap.afr_pct, color="#c1553b")
ax[0].set_title("AFR by capacity"); ax[0].set_xlabel("TB"); ax[0].set_ylabel("AFR %")
ax[1].bar(cap.index.astype(str), cap.drives, color="#1f7a6f")
ax[1].set_title("Fleet composition by capacity"); ax[1].set_xlabel("TB")
ax[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f"{v/1000:.0f}k"))
plt.tight_layout(); plt.show()
print(cap.to_string())

In [ ]:
# Drive age comes from power-on hours (smart_9_raw), read once per drive at its last observation.
age = (panel.select(["serial_number", "date", "smart_9_raw"])
            .filter(pl.col("smart_9_raw").is_not_null())
            .group_by("serial_number")
            .agg(poh = pl.col("smart_9_raw").max())
            .collect().to_pandas())
age["age_years"] = age.poh / 8766.0
age = age.merge(registry[["serial_number", "ever_failed", "n_days"]], on="serial_number")

bins   = [0, 0.5, 1, 2, 3, 4, 5, 6, 7, 8, 10, 100]
labels = ["<0.5", "0.5-1", "1-2", "2-3", "3-4", "4-5", "5-6", "6-7", "7-8", "8-10", "10+"]
age["bracket"] = pd.cut(age.age_years, bins=bins, labels=labels)

bath = (age.groupby("bracket", observed=True)
        .agg(drives=("serial_number", "size"),
             failures=("ever_failed", "sum"),
             drive_days=("n_days", "sum"))
        .assign(afr_pct=lambda d: (d.failures / d.drive_days * 365 * 100).round(2)))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(bath.index.astype(str), bath.afr_pct, "o-", color="#c1553b", lw=2)
ax.set_title("Bathtub curve: failure rate vs drive age\n"
             "(note: rates are enriched by the sampling, the SHAPE is what matters)")
ax.set_xlabel("age (years)"); ax.set_ylabel("failure rate %")
plt.tight_layout(); plt.show()
print(bath.to_string())

## 6. Topology and tome geometry

Read from the full-fleet table, because tome occupancy must reflect every drive in the slot
group - counting only sampled drives would understate occupancy and make the safety
invariant vacuous.

A tome is the set of drives sharing a `pod_slot_num` across a vault's 20 pods, erasure-coded
17+3 (also 16+4 and 15+5 on larger drives, where higher parity offsets longer rebuild times).
Two quantities define the invariant the agent must respect:

- **Redundancy set** - group by `(vault_id, pod_slot_num)`.
- **Parity budget** - how many members may be unavailable at once before data is lost.

In [ ]:
fleet = pl.scan_parquet(FLEET_GLOB)
ref_day = fleet.select(pl.col("date").max()).collect().item()

snap = (fleet.filter(pl.col("date") == ref_day)
             .select(["serial_number", "capacity_bytes", "vault_id", "pod_id", "pod_slot_num"])
             .collect())
print(f"reference day {ref_day}: {len(snap):,} drives")

tome_sizes = (snap.filter(pl.col("pod_slot_num").is_not_null())
                  .group_by(["vault_id", "pod_slot_num"])
                  .agg(n = pl.len()))
print("\n=== drives per redundancy set (vault_id, pod_slot_num) ===")
print(tome_sizes["n"].value_counts().sort("count", descending=True).head(8).to_pandas().to_string(index=False))

pods = snap.group_by("vault_id").agg(pods = pl.col("pod_id").n_unique())
print("\n=== pods per vault ===")
print(pods["pods"].value_counts().sort("count", descending=True).head(6).to_pandas().to_string(index=False))

In [ ]:
# Parity budget keyed on capacity, following Backblaze's published practice of raising parity
# on larger drives. This mapping is an explicit modelling assumption of the simulator.
def parity_for_tb(tb: float) -> int:
    if tb >= 14:  return 5      # 15+5
    if tb >= 10:  return 4      # 16+4
    return 3                    # 17+3

vc = (snap.group_by("vault_id")
          .agg(cap_tb = (pl.col("capacity_bytes").median() / 1e12).round(1),
               drives = pl.len())
          .to_pandas())
vc["parity"] = vc.cap_tb.map(parity_for_tb)

print("=== vaults by capacity and assigned parity budget ===")
print(vc.groupby(["cap_tb", "parity"]).agg(vaults=("vault_id", "size"),
                                           drives=("drives", "sum")).to_string())
print(f"\ntotal vaults {vc.vault_id.nunique():,} | parity mix:")
print(vc.parity.value_counts().sort_index().to_string())

## 7. Pre-failure trajectories

Align every failed drive on **days before its failure** and compare the median attribute
value against the healthy baseline. This is the visual test of whether early warning exists
at all: an attribute whose trajectory only diverges on the final day is useless for a 7-day
horizon, no matter how correlated it is with failure overall.

In [ ]:
failed_dates = (registry.loc[registry.ever_failed == 1, ["serial_number", "failure_date"]]
                .dropna())
print(f"failed drives with a failure date: {len(failed_dates):,}")

fd = pl.from_pandas(failed_dates).with_columns(
    pl.col("failure_date").cast(pl.Date))

traj = (panel.select(["serial_number", "date"] + raw_cols)
             .join(fd.lazy(), on="serial_number", how="inner")
             .with_columns(
                 dtf = (pl.col("failure_date") - pl.col("date")).dt.total_days())
             .filter((pl.col("dtf") >= 0) & (pl.col("dtf") <= 45))
             .collect())
print(f"pre-failure rows: {len(traj):,}")

healthy_serials = registry.loc[registry.ever_failed == 0, "serial_number"]

# Only the per-attribute medians are needed, so aggregate lazily rather than materialising
# every healthy row.
baseline = (panel.select(["serial_number"] + raw_cols)
                 .filter(pl.col("serial_number").is_in(healthy_serials.tolist()))
                 .select([pl.col(c).median().alias(c) for c in raw_cols])
                 .collect())
print("healthy baseline medians computed")

In [ ]:
focus = ["smart_5_raw", "smart_197_raw", "smart_198_raw", "smart_187_raw",
         "smart_1_raw", "smart_7_raw", "smart_194_raw", "smart_9_raw", "smart_199_raw"]
focus = [c for c in focus if c in raw_cols]

curves = (traj.group_by("dtf")
              .agg([pl.col(c).median().alias(c) for c in focus])
              .sort("dtf", descending=True).to_pandas())

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, c in zip(axes.ravel(), focus):
    ax.plot(curves.dtf, curves[c], color="#c1553b", lw=1.8, label="failing")
    base = baseline[c][0]
    if base is not None:
        ax.axhline(base, color="#1f7a6f", ls="--", lw=1.3, label="healthy median")
    ax.invert_xaxis()
    ax.set_title(c.replace("_raw", ""), fontsize=10)
    ax.set_xlabel("days before failure", fontsize=8)
    ax.legend(fontsize=7)
for ax in axes.ravel()[len(focus):]:
    ax.axis("off")
plt.suptitle("Median attribute trajectory approaching failure (x-axis reversed: failure at right)",
             y=1.01)
plt.tight_layout(); plt.show()

## 8. Anomaly onset and available lead time

For each failed drive, find the first day it showed a non-zero reallocated / pending /
uncorrectable sector count, and measure how long before failure that was.

This is the **single most important number for horizon selection**. The SMART-Z paper reports
a median first-anomaly-to-failure gap of 7 days with a maximum of 56, measured on a completely
different fleet (a Chinese video data centre, 2017-18). If Backblaze 2026 reproduces roughly
that median, the 7-day primary horizon is justified by two independent datasets rather than
by convention - and the fraction of drives with **zero** lead time sets a hard ceiling on
achievable recall that no model can exceed.

In [ ]:
onset_cols = [c for c in ["smart_5_raw", "smart_197_raw", "smart_198_raw"] if c in raw_cols]

onset = (traj.with_columns(
             anomaly = pl.any_horizontal([pl.col(c) > 0 for c in onset_cols]))
             .filter(pl.col("anomaly"))
             .group_by("serial_number")
             .agg(lead_days = pl.col("dtf").max())
             .to_pandas())

n_failed  = len(failed_dates)
n_warned  = len(onset)
print(f"failed drives                      : {n_failed:,}")
print(f"showed any sector anomaly in 45d   : {n_warned:,}  [{100*n_warned/n_failed:.1f}%]")
print(f"NEVER showed one (silent failures) : {n_failed-n_warned:,}  "
      f"[{100*(n_failed-n_warned)/n_failed:.1f}%]  <- ceiling on recall from these attributes")

if len(onset):
    print("\n=== lead time from first anomaly to failure (days) ===")
    print(onset.lead_days.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())
    print(f"\nSMART-Z reported median 7 days (max 56) on an independent fleet.")
    print(f"Backblaze Q1 2026 median here : {onset.lead_days.median():.0f} days")
    for h in (7, 14, 30):
        print(f"  caught with >= {h:2d}d lead : {(onset.lead_days >= h).mean()*100:5.1f}%")

In [ ]:
if len(onset):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].hist(onset.lead_days, bins=45, color="#1f7a6f", edgecolor="white", linewidth=0.4)
    ax[0].axvline(7, color="#c1553b", ls="--", lw=1.6, label="7d horizon")
    ax[0].axvline(onset.lead_days.median(), color="black", ls=":", lw=1.6,
                  label=f"median {onset.lead_days.median():.0f}d")
    ax[0].set_title("Warning lead time available"); ax[0].set_xlabel("days before failure")
    ax[0].legend(fontsize=8)

    s = np.sort(onset.lead_days.values)
    ax[1].plot(s, 1 - np.arange(len(s)) / len(s), color="#1f7a6f", lw=2)
    ax[1].axvline(7, color="#c1553b", ls="--", lw=1.4)
    ax[1].set_title("Fraction of failures with at least X days of warning")
    ax[1].set_xlabel("days"); ax[1].set_ylabel("fraction")
    plt.tight_layout(); plt.show()

## 9. Censoring

Drives leave the fleet constantly through migration and decommissioning, and Drive Stats does
**not** flag those departures. A drive that simply stops appearing did not necessarily fail.

This matters because the naive labelling - "no `failure=1`, therefore negative" - marks the
final days of a deliberately-drained drive as healthy. Those are exactly the days when a
drive being retired for looking suspicious would be showing degradation, so the naive label
actively teaches the model that pre-removal telemetry is fine. Notebook 03 marks these drives
right-censored and drops their trailing horizon window instead of labelling it.

In [ ]:
fleet_last = registry.last_seen.max()
registry["left_early"] = (registry.last_seen < fleet_last) & (registry.ever_failed == 0)

print(f"fleet last observed day : {fleet_last}")
print(f"drives leaving early without a failure flag: "
      f"{int(registry.left_early.sum()):,}  [{100*registry.left_early.mean():.1f}%]")

departures = (registry[registry.left_early]
              .groupby("last_seen").size().rename("departures").reset_index())
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(departures.last_seen, departures.departures, color="#8a6d3b", lw=1.2)
ax.set_title("Un-flagged departures per day (migrations / decommissions, not failures)")
ax.set_xlabel("last day seen"); ax.set_ylabel("drives")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

print("\nLargest single-day departures (bulk vault migrations):")
print(departures.nlargest(8, "departures").to_string(index=False))

## 10. Class prevalence

Sets the metric strategy. At a ~1.24% annualised failure rate, a 7-day positive window is
rare enough that accuracy is meaningless and ROC-AUC is misleadingly flattering, because the
enormous true-negative pool inflates it. Precision-recall AUC, and precision at a fixed
false-alarm budget, are the honest choices.

In [ ]:
reg = registry
fleet_afr = reg.ever_failed.sum() / reg.n_days.sum() * 365 * 100
print(f"fleet AFR (registry)            : {fleet_afr:.2f}%   [Backblaze publish 1.24% for Q1 2026]")

daily_hazard = reg.ever_failed.sum() / reg.n_days.sum()
print(f"per-drive-day failure probability: {daily_hazard:.6f}  (1 in {1/daily_hazard:,.0f})")
for h in (7, 14, 30):
    print(f"  approx {h:2d}-day positive prevalence: {daily_hazard*h*100:.3f}%  "
          f"(~1 positive per {1/(daily_hazard*h):,.0f} rows)")

print("\nConsequences for notebook 03 and 04:")
print("  - report PR-AUC, not ROC-AUC")
print("  - bootstrap CIs: the test window holds only ~250 failures")
print("  - calibrate probabilities; the healing policy thresholds on them directly")
print("  - reweight sampled metrics to fleet prevalence using cohort sample_weight")

## 11. Findings that drive notebook 03

Fill these in from the outputs above - they are the handoff to feature engineering.

1. **Vendor-conditional attributes.** `smart_187` / `smart_188` are Seagate-only, roughly
   two-thirds null fleet-wide. Handle with missingness indicators plus manufacturer as a
   feature. Never impute.

2. **Horizon choice.** Compare the measured median anomaly-to-failure lead time against
   SMART-Z's 7 days. Two independent fleets agreeing is a far stronger justification than
   convention.

3. **Silent failures.** The share of failed drives that never showed a sector anomaly is a
   hard ceiling on recall from those attributes alone, and the argument for adding
   trajectory-shape and error-rate features rather than relying on counters.

4. **Age.** If the bathtub curve is non-monotonic, include age explicitly.

5. **Censoring.** The early-departure fraction is the volume of rows that must be excluded
   from negative labels rather than assumed healthy.

6. **Tome geometry.** The redundancy set is `(vault_id, pod_slot_num)` and the parity budget
   is capacity-dependent. These define the safety invariant the reliability checker enforces.

7. **Imbalance.** Prevalence at the chosen horizon determines the metric suite and the
   expected-cost threshold, which is derived from failure/false-alarm costs rather than set
   to 0.5.